In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style("whitegrid")

df = pd.read_csv("youtube_videos.csv")

In [2]:
df.head()

,video_id,title,category,channel_tier,upload_day,upload_hour,duration_seconds,views,likes,comments,shares,watch_time_minutes,avg_view_duration_seconds,click_through_rate,channel_age_years,subscribers,tags_count,hashtags
0,vid_0000,I tried EVERY transformers tool — the WINNER s...,Music,Mid,Sunday,0,721,228102,9979,2810,1193,1091087,287,0.0100,1.3,441016,18.0,"[""ai"", ""tutorial"", ""trending""]"
1,vid_0001,What nobody tells you about AI image tools,Gaming,Small,Saturday,21,1894,25078,962,283,183,423818,1014,0.0230,0.8,25339,17.0,"[""tutorial""]"
2,vid_0002,What nobody tells you about machine learning p...,Gaming,Large,Monday,18,1413,1323949,6619,25177,16119,12952634,587,0.1022,4.8,2398429,NaN,"[""explained""]"
3,vid_0003,Is Claude worth it in 2025?,Gaming,Mid,Thursday,14,1335,187752,4936,2051,1797,2018334,645,0.0413,3.9,498144,13.0,"[""gaming"", ""trending"", ""explained"", ""tech""]"
4,vid_0004,URGENT: voice cloning AI is about to CHANGE FO...,Music,Small,Monday,4,3158,12583,777,98,30,262565,1252,0.0100,1.3,30039,10.0,"[""explained"", ""shorts""]"


## Min-Max Scaling Subscribers

Compress data scale between 0-1

In [3]:
from sklearn.preprocessing import MinMaxScaler

# create column called "subscribers min-maxed"
scaler = MinMaxScaler()
df["subscribers_scaled"] = scaler.fit_transform(df[["subscribers"]])


In [4]:
df["subscribers_scaled"].describe()

count    210.000000
mean       0.128853
std        0.265999
min        0.000000
25%        0.002555
50%        0.016794
75%        0.061946
max        1.000000
Name: subscribers_scaled, dtype: float64

## Standard Scaling 'views'
Since `views` is heavily skewed, Min-max would compress 95% of the data near 0. We don't want to use this scaling on large datasets

With standarization, values are now centred around 0 with most videos between -1 and 2, outliers beyond that (ex. 5)

In [6]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
df["views_zscore"] = standard_scaler.fit_transform(df[["views"]])
df[["views", "views_zscore"]].describe()


,views,views_zscore
count,2.100000e+02,2.100000e+02
mean,5.485440e+05,2.537653e-17
std,1.223751e+06,1.002389e+00
min,1.412000e+03,-4.481625e-01
25%,1.080400e+04,-4.404694e-01
50%,6.597350e+04,-3.952794e-01
75%,3.108772e+05,-1.946757e-01
max,7.038551e+06,5.316044e+00


## Robust Scaling

Z-score still uses the mean thats pulled by outliers. This pulls the mean/std distorting the Z-scale for everyone else.

Robust scaling uses the median and IQR instead.

The outlier is visible but it no longer collapses everyone else to near zero.

In [8]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()
df["views_robust"] = robust_scaler.fit_transform(df[["views"]])
df[["views", "views_robust"]].describe()


,views,views_robust
count,2.100000e+02,210.000000
mean,5.485440e+05,1.608176
std,1.223751e+06,4.078175
min,1.412000e+03,-0.215152
25%,1.080400e+04,-0.183853
50%,6.597350e+04,0.000000
75%,3.108772e+05,0.816147
max,7.038551e+06,23.236251


## Log Transform

real-world variables grow exponentially: a small number of cases dominate the scale which compresses everything else to the left 

log transform pulls the long tail back in.

`views`, `subscribers`, and `watch_time_minutes` are all skewed to the right. 

A small channel with 1k views and a large channel with 7m views are worlds apart but models see them on a linear scale.
- The gap between Small and Large compresses from 7m to 8 log units - a scale a model can reason about


In [9]:
df["views_log"] = np.log1p(df["views"])
df["views"]

0       228102
1        25078
2      1323949
3       187752
4        12583
        ...   
205       6441
206       4755
207     104852
208       3026
209     145045
Name: views, Length: 210, dtype: int64